### Regression model for predicting vote share


In [1]:
#Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import predict_election_model as pem
from pathlib import Path

#Configuration
from predict_election_model import EXCLUDE_COLS, WINNING_PARTIES , SELECTED_FEATURES

CWD = Path.cwd()
PROJECT_ROOT = CWD.parents[1]

DATA_DIR = PROJECT_ROOT / "processed-data" / "combined"

OUTPUT_DIR = PROJECT_ROOT / "models"/ "predict_election"/ "results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Getting the combined_data with selected features and conforming it into pandas dataframes.

#Combined data with selected features from election years: 2009, 2013, 2017 and 2021
path = DATA_DIR / "combined_features_2009_2021.csv"

df_combined_features_2009_2021 = pd.read_csv(path, sep=',')


In [3]:
# Getting the preprocessed election result data from 2025
path = DATA_DIR / "combined_2021_dem_2025_elec.csv"

df_combined_2021_dem_2025_elec = pd.read_csv(path, sep=',')

In [4]:
df_combined_2021_dem_2025_elec

,Gruppe,Municipality,A_vote_share,C_vote_share,F_vote_share,V_vote_share,Ø_vote_share,winning_party,winning_votes,Sex_Female,...,Income_Median_normalized,Children_0_17,Youth_18_29,WorkingAge_30_64,Seniors_65_plus,Education_Long_higher_education,Education_Primary_lower_secondary,Education_Vocational_training,Households_with_car,People_on_benefits
0,101001,København,15.623180,18.156669,16.788002,7.367501,15.972627,C,18.156669,52.891659,...,71.385246,16.533865,26.018507,45.836011,11.611618,37.800951,10.259163,12.544088,37.145699,3.232232
1,101002,København,13.283406,20.141900,15.116279,8.927868,13.716989,C,20.141900,53.354022,...,69.726746,16.782048,21.986154,44.127477,17.104321,34.531653,12.918934,14.866800,44.521982,4.105992
2,101003,København,13.077513,15.717575,18.741366,7.029931,18.112049,F,18.741366,51.785940,...,75.051252,19.222517,22.523034,47.248517,11.005932,40.057727,9.563929,11.506358,39.805953,2.189827
3,101005,København,14.209763,14.644756,17.883035,6.778637,18.595940,Ø,18.595940,51.759808,...,67.236974,15.529204,26.510660,44.152895,13.807241,30.339084,13.838489,16.418798,35.995702,4.352214
4,101006,København,13.894737,15.013158,15.986842,6.605263,18.250000,Ø,18.250000,51.605249,...,68.155969,16.690146,25.950916,44.479870,12.879069,31.825453,13.917526,16.743690,38.789585,5.034478
5,101007,København,16.681495,9.326809,17.986358,5.308422,19.543298,Ø,19.543298,51.139152,...,63.950418,18.703120,21.079565,45.222573,14.994742,19.546082,22.877114,23.136003,41.332771,7.584998
6,101008,København,14.163004,10.207751,18.138234,5.852976,21.634039,Ø,21.634039,51.050739,...,100.000000,17.899712,23.553191,48.485569,10.061528,31.835475,13.868351,16.757591,40.116938,5.050085
7,101009,København,12.635507,9.952017,14.963569,8.690243,18.873289,Ø,18.873289,49.723134,...,97.406394,20.945206,29.985191,42.769300,6.300303,28.984615,13.308718,15.577436,40.041128,3.319168
8,101011,København,10.494162,14.349715,16.576161,6.204181,22.237307,Ø,22.237307,46.191392,...,75.211873,12.695361,30.666075,47.231672,9.406892,36.993384,13.887808,9.996108,33.907904,6.152936
9,101012,København,12.087327,21.849485,14.288250,9.407171,15.459709,C,21.849485,51.710985,...,81.619269,18.962511,19.158675,46.120314,15.758500,42.801868,8.819213,10.433622,50.460271,1.514821


In [9]:
temp_val_results = pem.validate_temporal_lag(df_combined_features_2009_2021, selected_features=SELECTED_FEATURES)

----------------------------------------------------------------------
Temporal lag validation
Testing: Can demographics from year N predict election at N+4?
----------------------------------------------------------------------

----------------------------------------------------------------------
TEST: 2009 demographics → 2013 election
----------------------------------------------------------------------
Matched 58 polling areas

Performance Summary:
  Party A: MAE=3.32%, R²=0.304
  Party V: MAE=2.89%, R²=-0.015
  Party C: MAE=5.96%, R²=0.512
  Party F: MAE=11.47%, R²=-34.253
  Party Ø: MAE=7.95%, R²=-1.781

----------------------------------------------------------------------
TEST: 2013 demographics → 2017 election
----------------------------------------------------------------------
Matched 58 polling areas

Performance Summary:
  Party A: MAE=3.48%, R²=0.399
  Party V: MAE=2.71%, R²=-0.302
  Party C: MAE=5.07%, R²=0.600
  Party F: MAE=2.06%, R²=-2.148
  Party Ø: MAE=3.21%, R²=

In [10]:
lag_results = temp_val_results["raw"]
df_temporal_metrics = temp_val_results["metrics_df"]
df_temporal_summary = temp_val_results["summary_df"]

In [11]:
df_temporal_metrics

,period,party,mae,rmse,r2
0,2009→2013,A,3.322723,4.217365,0.304246
1,2009→2013,V,2.887732,3.796336,-0.015135
2,2009→2013,C,5.957475,7.178003,0.511500
3,2009→2013,F,11.465247,11.628772,-34.253479
4,2009→2013,Ø,7.949079,8.902551,-1.781441
5,2013→2017,A,3.475729,4.258805,0.398614
6,2013→2017,V,2.706835,3.085518,-0.301713
7,2013→2017,C,5.072688,6.669853,0.599677
8,2013→2017,F,2.056679,2.535665,-2.147842
9,2013→2017,Ø,3.210834,3.967813,0.404771


In [12]:
# Making a line plot: MAE over time
save_path = OUTPUT_DIR / 'temp_lag_val_MAE'

pem.plot_temporal_mae(
    df_temporal_metrics,
    figsize=(10, 6),
    title="Temporal Lag Validation: Mean Absolute Error",
    save_path=save_path,
)

c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:602: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
#Making a heatmap with R² scores from the temporal lag validation

r2_matrix = df_temporal_metrics.pivot(
    index="party",
    columns="period",
    values="r2"
)

save_path = OUTPUT_DIR / 'temp_lag_val_r2'

pem.plot_temporal_r2_heatmap(
    r2_matrix,
    figsize=(7, 4),
    cmap="RdYlGn",
    title="Temporal Lag Validation: R² Scores",
    save_path=save_path,
)

c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:660: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
same_year_val_results = pem.validate_same_year(df_combined_features_2009_2021, selected_features=SELECTED_FEATURES)

----------------------------------------------------------------------
SAME-YEAR VALIDATION: Train 2009-2017 → Test 2021
(Using 2021 demographics → 2021 election)
----------------------------------------------------------------------

Training: 174 samples from 2009, 2013, 2017
Testing: 58 samples from 2021

----------------------------------------------------------------------
Party A
----------------------------------------------------------------------
MAE:  7.11%
RMSE: 8.16%
R²:   -2.135

----------------------------------------------------------------------
Party V
----------------------------------------------------------------------
MAE:  2.63%
RMSE: 3.05%
R²:   -0.628

----------------------------------------------------------------------
Party C
----------------------------------------------------------------------
MAE:  4.95%
RMSE: 6.41%
R²:   0.583

----------------------------------------------------------------------
Party F
------------------------------------------------

In [16]:
#Retriving the specific 
results = same_year_val_results["party_results"]
all_predictions = same_year_val_results["predictions_wide"]
df_long = same_year_val_results["predictions_long"]
accuracy = same_year_val_results["accuracy"]


In [17]:
# Scatter plot: Predicted vs Actual (per party)

for party in WINNING_PARTIES:
    pem.plot_predicted_vs_actual(
        df=df_long[df_long["party"] == party],
        party=party,
        title=f"Same-Year Validation (2021): Party {party}",
        save_path= OUTPUT_DIR / f"same_year_scatter_{party}"
    )


c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-intera

In [18]:
final_models, df_cv_r2 = pem.train_final_models(df_combined_features_2009_2021, selected_features=SELECTED_FEATURES)

----------------------------------------------------------------------
Final models: Training on 2009-2021
----------------------------------------------------------------------

Training Party A...
  R² (CV): 0.555

Training Party V...
  R² (CV): 0.400

Training Party C...
  R² (CV): 0.623

Training Party F...
  R² (CV): 0.874

Training Party Ø...
  R² (CV): 0.735


In [19]:
save_path = OUTPUT_DIR / 'final_model_cv_r2'

pem.plot_cv_r2(
    df_cv_r2,
    figsize=(6, 4),
    title="Final Models: Cross-Validated R² (2009-2021)",
    save_path=save_path,
)

c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:774: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
final_predictions_results = pem.predict_2025_with_proxy(df_combined_2021_dem_2025_elec, final_models, parties=WINNING_PARTIES, exclude_cols=EXCLUDE_COLS)

----------------------------------------------------------------------
2025 PREDICTION USING 2021 DEMOGRAPHICS AS PROXY
----------------------------------------------------------------------

OBS: Using 2021 demographics to predict 2025 election
     Assumes demographics are stable over time

Predicting Party A...
  MAE: 9.01%
  R²:  -5.758
  Predicted mean: 22.8%
  Actual mean:    13.9%

Predicting Party V...
  MAE: 2.63%
  R²:  -1.804
  Predicted mean: 8.1%
  Actual mean:    5.6%

Predicting Party C...
  MAE: 3.62%
  R²:  0.737
  Predicted mean: 14.8%
  Actual mean:    13.7%

Predicting Party F...
  MAE: 6.22%
  R²:  -2.189
  Predicted mean: 10.7%
  Actual mean:    16.6%

Predicting Party Ø...
  MAE: 2.99%
  R²:  0.672
  Predicted mean: 20.8%
  Actual mean:    22.0%

----------------------------------------------------------------------
2025 WINNER PREDICTION ACCURACY: 50.0%
----------------------------------------------------------------------

Confusion Matrix:
Predicted   A   C   

In [21]:
predictions_2025 = final_predictions_results["predictions_wide"]
df_2025_long = final_predictions_results["predictions_long"]
df_2025_metrics = final_predictions_results["metrics"]
confusion = final_predictions_results["confusion"]
accuracy = final_predictions_results["accuracy"]

In [22]:
save_path = OUTPUT_DIR / 'final_model_cv_r2'

pem.plot_predicted_vs_actual(
    df_2025_long,
    party,
    figsize=(5, 5),
    title=None,
    save_path=None,
)

c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# Scatter plot: Predicted vs Actual (per party) for 2025 election

for party in WINNING_PARTIES:
    pem.plot_predicted_vs_actual(
        df=df_2025_long[df_2025_long["party"] == party],
        party=party,
        title=f"2025 Predicted vs. Actual: Party {party}",
        save_path= OUTPUT_DIR / f"final_2025_scatter_{party}"
    )


c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:727: UserWarning: FigureCanvasPgf is non-intera

In [23]:
save_path = OUTPUT_DIR / '2025_confusion_matrix'

pem.plot_confusion_matrix(
    confusion,
    figsize=(5, 4),
    cmap="Blues",
    title="2025 Election: Winner Prediction Confusion Matrix",
    save_path=save_path,
)

c:\Users\Johanne Auener\Desktop\DAMIN A2\local-elections-damin2025\models\predict_election\predict_election_model.py:822: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
